# GPU Final Notebook - Capitals

Modernized final capitals run using behavior-specific capitals SAEs plus contrast-target attribution graphs, all-position full latent swaps, graph-positive sparse swaps, and separate `capitals_final_*` outputs.

This notebook trains or resumes final capitals-specific SAEs in `sae_checkpoints_capitals_final_train`, then saves distinct `capitals_final_*` graph/intervention outputs.


## Step 1 - Mount Drive and fetch code


In [ ]:
# Step 1: Mount Google Drive and fetch project code
# GitHub is the primary source for code. Drive project.zip is kept as a backup.
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

import os
import subprocess

repo_url = "https://github.com/evey-dev/test_run.git"
repo_dir = "/content/test_run"
use_github_code = True

def run_cmd(cmd):
    print("$", " ".join(cmd))
    subprocess.run(cmd, check=True)

github_ok = False
if use_github_code:
    try:
        clone_dir = repo_dir
        if os.path.isdir(os.path.join(clone_dir, ".git")):
            run_cmd(["git", "-C", clone_dir, "pull", "--ff-only"])
        else:
            if os.path.exists(clone_dir) and os.listdir(clone_dir):
                clone_dir = "/content/test_run_github"
            if os.path.isdir(os.path.join(clone_dir, ".git")):
                run_cmd(["git", "-C", clone_dir, "pull", "--ff-only"])
            elif os.path.exists(clone_dir) and os.listdir(clone_dir):
                raise RuntimeError(f"{clone_dir} exists but is not a git repo")
            else:
                run_cmd(["git", "clone", "--depth", "1", repo_url, clone_dir])
        os.chdir(clone_dir)
        github_ok = True
        print(f"Using GitHub checkout: {os.getcwd()}")
    except Exception as exc:
        print("GitHub checkout failed; falling back to Drive project.zip.")
        print(repr(exc))

if not github_ok:
    zip_path = "/content/drive/MyDrive/mphil-project/project.zip"
    if not os.path.exists(zip_path):
        raise FileNotFoundError(f"Could not find {zip_path}. Fix GitHub access or upload project.zip.")
    print(f"Extracting backup zip {zip_path}...")
    run_cmd(["unzip", "-q", "-o", zip_path, "-d", "/content/"])
    for candidate in ["/content/test_run", "/content/mphil_project/test_run", "/content"]:
        if os.path.isdir(os.path.join(candidate, "src")) and os.path.isdir(os.path.join(candidate, "configs")):
            os.chdir(candidate)
            break
    else:
        raise FileNotFoundError("Could not find extracted project root containing src/ and configs/.")
    print(f"Using Drive backup project: {os.getcwd()}")

print(f"Current working directory: {os.getcwd()}")
!ls -l


## Step 2 - Install dependencies


In [ ]:
# Step 2: Install dependencies
!pip install --upgrade "transformers>=4.51.0" accelerate
!pip install -e .


## Step 3 - Generate capitals dataset


In [ ]:
# Step 3: Generate capitals dataset
!python data/generate_datasets.py --capitals
print("Dataset files:")
!ls -lh data/capitals_data.csv || true
!ls -lh data/*capital* || true


## Step 4 - Capture capitals-only activations


In [ ]:
# Capture capitals-only MLP activations for final capitals SAE training
!python src/capture_activations.py \
  --output-dir mechanistic_data_capitals_final_train \
  --behaviours capitals \
  --layers 4 8 12 16 20 24 28 \
  --seed 787

print("Final capitals activation bundle:")
!ls -lh mechanistic_data_capitals_final_train


## Step 5 - Train final capitals-specific SAEs


In [ ]:
# Train final capitals-specific SAEs for all selected layers
!python src/train.py \
  --config configs/sae_capitals_final_train_config.yaml \
  --drive-dir /content/drive/MyDrive/mphil-project/mechanistic_data/sae_checkpoints_capitals_final_train

print("Final capitals local checkpoints:")
!ls -lh mechanistic_data/sae_checkpoints_capitals_final_train
print("Final capitals Drive checkpoints:")
!ls -lh /content/drive/MyDrive/mphil-project/mechanistic_data/sae_checkpoints_capitals_final_train


## Step 6 - Sanity check the Dallas/Oakland minimal pair


In [ ]:
# Confirm token positions before all-position swaps.
# Dallas -> Austin and Oakland -> Sacramento should be comparable prompts.
!python src/intervention.py \
  --mode inhibit \
  --prompt "Fact: The capital of the state containing Dallas is named" \
  --print-tokens

!python src/intervention.py \
  --mode inhibit \
  --prompt "Fact: The capital of the state containing Oakland is named" \
  --print-tokens


## Step 7 - Rebuild contrast attribution graphs


In [ ]:
# Dallas graph: Austin vs Sacramento.
!python src/attribution_graph.py \
  --prompt "Fact: The capital of the state containing Dallas is named" \
  --target "Austin" \
  --contrast-target "Sacramento" \
  --layers 4 8 12 16 20 24 28 \
  --sae-config configs/sae_capitals_final_train_config.yaml \
  --top-k-nodes 20 --top-k-edges 30 \
  --output-json outputs/capitals_final_dallas_austin_vs_sacramento_graph.json \
  --output-html outputs/capitals_final_dallas_austin_vs_sacramento_graph.html \
  --output-mermaid outputs/capitals_final_dallas_austin_vs_sacramento_graph.md


In [ ]:
# Oakland graph: Sacramento vs Austin.
!python src/attribution_graph.py \
  --prompt "Fact: The capital of the state containing Oakland is named" \
  --target "Sacramento" \
  --contrast-target "Austin" \
  --layers 4 8 12 16 20 24 28 \
  --sae-config configs/sae_capitals_final_train_config.yaml \
  --top-k-nodes 20 --top-k-edges 30 \
  --output-json outputs/capitals_final_oakland_sacramento_vs_austin_graph.json \
  --output-html outputs/capitals_final_oakland_sacramento_vs_austin_graph.html \
  --output-mermaid outputs/capitals_final_oakland_sacramento_vs_austin_graph.md


In [ ]:
# Persist final capitals graphs immediately.
import os, shutil, glob

drive_out = '/content/drive/MyDrive/mphil-project/outputs'
os.makedirs(drive_out, exist_ok=True)
for src in glob.glob('/content/test_run/outputs/capitals_final_*_graph.*'):
    if os.path.isfile(src):
        shutil.copy2(src, drive_out)
        print('Copied', os.path.basename(src))
print('Done!')


## Step 8 - Component and sparse controls


In [ ]:
# All-position MLP knockout on Dallas/Austin.
!python src/intervention.py \
  --mode inhibit \
  --prompt "Fact: The capital of the state containing Dallas is named" \
  --target-token "Austin, Sacramento" \
  --layers 4 8 12 16 20 24 28 \
  --sae-config configs/sae_capitals_final_train_config.yaml \
  --full-knockout \
  --knockout-component mlp \
  --positions all \
  --layer-scan \
  --output outputs/capitals_final_knockout_mlp_allpos_dallas.json


In [ ]:
# Progressive sparse ablation of Dallas graph-positive features.
!python src/intervention.py \
  --mode inhibit \
  --prompt "Fact: The capital of the state containing Dallas is named" \
  --target-token "Austin, Sacramento" \
  --layers 4 8 12 16 20 24 28 \
  --sae-config configs/sae_capitals_final_train_config.yaml \
  --graph-json outputs/capitals_final_dallas_austin_vs_sacramento_graph.json \
  --graph-feature-sign positive \
  --scan \
  --positions all \
  --output outputs/capitals_final_scan_dallas_positive_allpos.json


## Step 9 - Full latent minimal-pair swaps


In [ ]:
# Patch Oakland/Sacramento activations into Dallas/Austin.
# Desired direction: Austin falls and Sacramento rises.
!python src/intervention.py \
  --mode swap \
  --source-prompt "Fact: The capital of the state containing Oakland is named" \
  --prompt "Fact: The capital of the state containing Dallas is named" \
  --layers 4 8 12 16 20 24 28 \
  --sae-config configs/sae_capitals_final_train_config.yaml \
  --target-token "Austin, Sacramento" \
  --positions all \
  --layer-scan \
  --output outputs/capitals_final_swap_full_oakland_to_dallas.json


In [ ]:
# Reverse direction: patch Dallas/Austin activations into Oakland/Sacramento.
# Desired direction: Sacramento falls and Austin rises.
!python src/intervention.py \
  --mode swap \
  --source-prompt "Fact: The capital of the state containing Dallas is named" \
  --prompt "Fact: The capital of the state containing Oakland is named" \
  --layers 4 8 12 16 20 24 28 \
  --sae-config configs/sae_capitals_final_train_config.yaml \
  --target-token "Austin, Sacramento" \
  --positions all \
  --layer-scan \
  --output outputs/capitals_final_swap_full_dallas_to_oakland.json


## Step 10 - Sparse graph-feature swaps


In [ ]:
# Sparse Oakland -> Dallas, using Oakland graph-positive Sacramento features.
!python src/intervention.py \
  --mode swap \
  --source-prompt "Fact: The capital of the state containing Oakland is named" \
  --prompt "Fact: The capital of the state containing Dallas is named" \
  --layers 4 8 12 16 20 24 28 \
  --sae-config configs/sae_capitals_final_train_config.yaml \
  --graph-json outputs/capitals_final_oakland_sacramento_vs_austin_graph.json \
  --graph-feature-sign positive \
  --target-token "Austin, Sacramento" \
  --positions all \
  --output outputs/capitals_final_swap_sparse_oakland_to_dallas.json


In [ ]:
# Sparse Dallas -> Oakland, using Dallas graph-positive Austin features.
!python src/intervention.py \
  --mode swap \
  --source-prompt "Fact: The capital of the state containing Dallas is named" \
  --prompt "Fact: The capital of the state containing Oakland is named" \
  --layers 4 8 12 16 20 24 28 \
  --sae-config configs/sae_capitals_final_train_config.yaml \
  --graph-json outputs/capitals_final_dallas_austin_vs_sacramento_graph.json \
  --graph-feature-sign positive \
  --target-token "Austin, Sacramento" \
  --positions all \
  --output outputs/capitals_final_swap_sparse_dallas_to_oakland.json


## Step 11 - Persist final capitals outputs


In [ ]:
# Copy final capitals graphs/results to Drive
import os, shutil, glob

drive_out = '/content/drive/MyDrive/mphil-project/outputs'
os.makedirs(drive_out, exist_ok=True)
for src in glob.glob('/content/test_run/outputs/capitals_final_*'):
    if os.path.isfile(src):
        shutil.copy2(src, drive_out)
        print('Copied', os.path.basename(src))
print('Done!')
